# Análisis OLAP - PostgreSQL & Hive

Tres análisis sobre los datos del restaurante almacenados en PostgreSQL.  
Los resultados se persisten en Apache Hive como tablas permanentes.

| # | Análisis | Tabla Hive |
|---|---|---|
| 1 | Tendencias de consumo | `restaurantes.tendencias_consumo` |
| 2 | Horarios pico | `restaurantes.horarios_pico` |
| 3 | Crecimiento mensual | `restaurantes.crecimiento_mensual` |

## 0. Configuración

In [ ]:
PG_HOST     = "host.docker.internal"
PG_PORT     = 4001
PG_DB       = "restaurantes_db"
PG_USER     = "postgres"
PG_PASSWORD = "postgres"
JDBC_URL    = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"
JDBC_PROPS  = {"user": PG_USER, "password": PG_PASSWORD, "driver": "org.postgresql.Driver"}

HIVE_DB = "restaurantes"

## 1. Sesión de Spark

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("OLAP Restaurantes — PostgreSQL")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} listo.")

## 2. Lectura de tablas desde PostgreSQL

In [ ]:
from pyspark.sql.functions import col

def read_table(table):
    return spark.read.jdbc(url=JDBC_URL, table=table, properties=JDBC_PROPS)

pedidos       = read_table("pedidos")
item_pedido   = read_table("item_pedido")
platos        = read_table("platos")
menus         = read_table("menus")
reservaciones = read_table("reservaciones")

print(f"pedidos:       {pedidos.count():,} filas")
print(f"item_pedido:   {item_pedido.count():,} filas")
print(f"platos:        {platos.count():,} filas")
print(f"menus:         {menus.count():,} filas")
print(f"reservaciones: {reservaciones.count():,} filas")

## 3. Base de datos en Hive

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {HIVE_DB}")
spark.sql(f"USE {HIVE_DB}")
print(f"Base de datos '{HIVE_DB}' lista.")
spark.sql("SHOW DATABASES").show()

---
## Análisis 1 — Tendencias de consumo
Platos más pedidos globalmente y top 5 por categoría (excluyendo pedidos cancelados).

In [ ]:
from pyspark.sql.functions import sum as spark_sum, desc, rank
from pyspark.sql.window import Window

items = item_pedido.join(
    pedidos.select(col("id").alias("pedido_id_join"), col("estado")),
    item_pedido.pedido_id == col("pedido_id_join"), how="left"
).filter(col("estado") != "cancelado") \
 .select(col("plato_id"), col("cantidad"))

items_platos = items.join(
    platos.select(col("id").alias("plato_id_join"), "nombre_plato", "precio", "menu_id"),
    items.plato_id == col("plato_id_join"), how="left"
)

items_platos_menus = items_platos.join(
    menus.select(col("id").alias("menu_id_join"), col("nombre_menu").alias("categoria")),
    items_platos.menu_id == col("menu_id_join"), how="left"
)

window_cat = Window.partitionBy("categoria").orderBy(desc("total_pedido"))
top_por_categoria = (
    items_platos_menus
    .groupBy("categoria", "nombre_plato", "precio")
    .agg(spark_sum("cantidad").alias("total_pedido"))
    .orderBy(desc("total_pedido"))
    .withColumn("rank", rank().over(window_cat))
    .filter(col("rank") <= 5)
    .orderBy("categoria", "rank")
)

print("Top 5 platos por categoría:")
top_por_categoria.show(50, truncate=False)


In [ ]:
data = top_por_categoria.collect()
df_final = spark.createDataFrame(data, top_por_categoria.schema)
df_final.write.mode("overwrite").saveAsTable(f"{HIVE_DB}.tendencias_consumo")
print("Guardado en Hive: tendencias_consumo")
spark.sql(f"SELECT * FROM {HIVE_DB}.tendencias_consumo LIMIT 5").show(truncate=False)

---
## Análisis 2 — Horarios pico
Distribución de reservaciones activas y completadas por hora del día.

In [ ]:
from pyspark.sql.functions import hour, count, asc

horarios_pico = (
    reservaciones
    .filter(col("estado").isin("activa", "completada"))
    .withColumn("hora", hour(col("hora_reservacion")))
    .groupBy("hora")
    .agg(count("*").alias("total_reservaciones"))
    .orderBy(asc("hora"))
)

print("Reservaciones por hora del día:")
horarios_pico.show(24, truncate=False)

hora_pico = horarios_pico.orderBy(desc("total_reservaciones")).first()
print(f"→ Hora pico: {hora_pico['hora']}:00 con {hora_pico['total_reservaciones']} reservaciones")


In [ ]:
data = horarios_pico.collect()
df_final = spark.createDataFrame(data, horarios_pico.schema)
df_final.write.mode("overwrite").saveAsTable(f"{HIVE_DB}.horarios_pico")
print("Guardado en Hive: horarios_pico")
spark.sql(f"SELECT * FROM {HIVE_DB}.horarios_pico ORDER BY hora").show(truncate=False)

---
## Análisis 3 — Crecimiento mensual
Volumen de pedidos agrupado por mes con variación porcentual mes a mes.

In [ ]:
from pyspark.sql.functions import date_format, lag, round as spark_round, coalesce, lit

window_mes = Window.orderBy("anio_mes")
crecimiento_variacion = (
    pedidos
    .withColumn("anio_mes", date_format(col("fecha_orden"), "yyyy-MM"))
    .groupBy("anio_mes")
    .agg(count("*").alias("total_pedidos"))
    .orderBy(asc("anio_mes"))
    .withColumn("mes_anterior", lag("total_pedidos", 1).over(window_mes))
    .withColumn(
        "variacion_pct",
        coalesce(
            spark_round(
                ((col("total_pedidos") - col("mes_anterior")) / col("mes_anterior")) * 100, 2
            ),
            lit(0.0)
        )
    )
)

print("Variación porcentual mes a mes:")
crecimiento_variacion.show(50, truncate=False)


In [ ]:
data = crecimiento_variacion.collect()
df_final = spark.createDataFrame(data, crecimiento_variacion.schema)
df_final.write.mode("overwrite").saveAsTable(f"{HIVE_DB}.crecimiento_mensual")
print("Guardado en Hive: crecimiento_mensual")
spark.sql(f"SELECT * FROM {HIVE_DB}.crecimiento_mensual ORDER BY anio_mes").show(truncate=False)

---
## Resumen de tablas en Hive

In [ ]:
print(f"Tablas en la base de datos '{HIVE_DB}':")
spark.sql(f"SHOW TABLES IN {HIVE_DB}").show(truncate=False)

spark.stop()
print("Sesión cerrada.")